<a href="https://colab.research.google.com/github/chrisanuo/chrisanuo/blob/main/C_N_Ratio_split_plot_analysis_for_bulk_soil_POM_MAOM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%load_ext rpy2.ipython

In [2]:
%%R

install.packages(c(
  "readxl", "dplyr", "lme4", "lmerTest",
  "emmeans", "multcomp", "multcompView",
  "openxlsx"
))

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
also installing the dependencies ‘rbibutils’, ‘zoo’, ‘Rdpack’, ‘minqa’, ‘nloptr’, ‘reformulas’, ‘RcppEigen’, ‘numDeriv’, ‘estimability’, ‘mvtnorm’, ‘TH.data’, ‘sandwich’

trying URL 'https://cran.rstudio.com/src/contrib/rbibutils_2.4.1.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/zoo_1.8-15.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/Rdpack_2.6.6.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/minqa_1.2.8.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/nloptr_2.2.1.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/reformulas_0.4.4.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/RcppEigen_0.3.4.0.2.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/numDeriv_2016.8-1.1.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/estimability_2.0.0.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/mvtnorm_1.4-1.tar.gz'
trying URL 'https://c

In [3]:
from google.colab import files
uploaded = files.upload()

Saving Updated_bulk_POM_MAOM_C_N.xlsx to Updated_bulk_POM_MAOM_C_N.xlsx


In [5]:
%%R

library(readxl)
library(dplyr)
library(lme4)
library(lmerTest)
library(emmeans)
library(multcomp)
library(multcompView)
library(openxlsx)

file_path <- "Updated_bulk_POM_MAOM_C_N.xlsx"

sheets <- excel_sheets(file_path)

data_all <- lapply(sheets, function(sh) {
  read_excel(file_path, sheet = sh) %>%
    mutate(Depth = sh)
}) %>%
  bind_rows()

data_all <- data_all %>%
  rename(
    Tillage = Treatments,
    Rotation = `crop-rotation`,
    BulkCN = `Bulk C/N`,
    POMCN = `POM C/N`,
    MAOMCN = `MAOM C/N`
  ) %>%
  mutate(
    Rep = factor(Rep),
    Depth = factor(Depth, levels = c("0-5", "5-15", "15-30", "30-50", "50-75", "75-100")),
    Tillage = recode(Tillage, `MB plow` = "Moldboard plow"),
    Tillage = factor(Tillage, levels = c("No-till", "Chisel plow", "Moldboard plow")),
    Rotation = recode(
      Rotation,
      `C-C` = "Continuous corn",
      `C-B` = "Corn-soybean",
      `B-B` = "Continuous soybean"
    ),
    Rotation = factor(Rotation, levels = c("Continuous corn", "Corn-soybean", "Continuous soybean"))
  )

variables <- c("BulkCN", "POMCN", "MAOMCN")

anova_results <- list()
lsd_results <- list()

for (var in variables) {
  for (dep in levels(data_all$Depth)) {

    dat <- data_all %>%
      filter(Depth == dep) %>%
      filter(!is.na(.data[[var]]))

    if (nrow(dat) == 0) next

    model <- lmer(
      as.formula(paste(var, "~ Rotation * Tillage + (1|Rep) + (1|Rep:Rotation)")),
      data = dat
    )

    anova_tbl <- anova(model, type = 3) %>%
      as.data.frame() %>%
      mutate(
        Variable = var,
        Depth = dep,
        Effect = rownames(.)
      ) %>%
      dplyr::select(Variable, Depth, Effect, everything())

    anova_results[[paste(var, dep, sep = "_")]] <- anova_tbl

    emm <- emmeans(model, ~ Rotation * Tillage)

    lsd <- cld(
      emm,
      adjust = "none",
      Letters = letters,
      alpha = 0.05
    ) %>%
      as.data.frame() %>%
      mutate(
        Variable = var,
        Depth = dep
      ) %>%
      dplyr::select(Variable, Depth, Rotation, Tillage, emmean, SE, df, lower.CL, upper.CL, .group)

    lsd_results[[paste(var, dep, sep = "_")]] <- lsd
  }
}

anova_out <- bind_rows(anova_results)
lsd_out <- bind_rows(lsd_results)

wb <- createWorkbook()

addWorksheet(wb, "ANOVA")
writeData(wb, "ANOVA", anova_out)

addWorksheet(wb, "Fisher_LSD")
writeData(wb, "Fisher_LSD", lsd_out)

saveWorkbook(wb, "CN_ratio_splitplot_ANOVA_Fisher_LSD.xlsx", overwrite = TRUE)

boundary (singular) fit: see help('isSingular')
Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed
boundary (singular) fit: see help('isSingular')
Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed
boundary (singular) fit: see help('isSingular')
Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed
boundary (singular) fit: see help('isSingular')
Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed
Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed
boundary (singular) fit: see help('isSingular')
Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed
boundary (singular) fit: see help('isSingular')
Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed
boundary (singular) fit: see help('isSingular')
Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed
boundary (singular) fit: see help('isSin

In [6]:
from google.colab import files
files.download("CN_ratio_splitplot_ANOVA_Fisher_LSD.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>